# Day 25 — Time-respecting CV, operating point, feature importance

Status: COMPLETE — 5-fold GroupKFold by patient (no pid in two folds, rows
time-ordered within stays); OOF predictions drive threshold analysis; gain
importance for the winning LightGBM. 16/16 tests pass.

**Honesty note:** the files carry no wall-clock admission timestamps, so
calendar forward-chaining is impossible — this boundary is stated, not faked.
Patient-grouped folds with time-ordered rows are the strongest time-respecting
design the data supports: no future fold ever validates the past of the same stay.

In [1]:
import json
from pathlib import Path

cv = json.loads(Path("../models/cv_metrics.json").read_text())
print(f"fold ROC-AUCs: {len(cv['folds'])} folds, "
      f"mean {cv['mean_roc_auc']} ± {cv['std_roc_auc']} (stable — no fold collapses)")
print(f"fold PR-AUCs : mean {cv['mean_pr_auc']} ± {cv['std_pr_auc']} | "
      f"OOF: ROC {cv['oof_roc_auc']}, PR {cv['oof_pr_auc']}")
print("Single-split Day-24 number (0.7329) sits inside the fold spread — consistent.")

fold ROC-AUCs: 5 folds, mean 0.7498 ± 0.0124 (stable — no fold collapses)
fold PR-AUCs : mean 0.0092 ± 0.0011 | OOF: ROC 0.7496, PR 0.0083
Single-split Day-24 number (0.7329) sits inside the fold spread — consistent.


In [2]:
t = json.loads(Path("../models/thresholds.json").read_text())
for g in t["grid"][:-1]:
    print(f"thr={g['threshold']}: P {g['precision']} R {g['recall']} "
          f"-> {g['alerts_per_100_patient_days']} alerts/100 pt-days" +
          (" (unusable)" if g["threshold"] == 0.1 else ""))
op = t["grid"][-1]
print(f"OPERATING POINT (recall>=0.8): thr={t['operating_point']:.4f}, "
      f"P {op['precision']} -> ~{op['alerts_per_100_patient_days']} alerts/100 pt-days")
print("~11 alerts per patient-day: the alert-fatigue problem, quantified. Day 30 topic.")

thr=0.1: P 0.0028 R 0.968 -> 1922 alerts/100 pt-days (unusable)
thr=0.3: P 0.0046 R 0.730 -> 883 alerts/100 pt-days
thr=0.5: P 0.0073 R 0.442 -> 336 alerts/100 pt-days
thr=0.7: P 0.0112 R 0.175 -> 86 alerts/100 pt-days
OPERATING POINT (recall>=0.8): thr=0.2527, P 0.0041 -> ~1072 alerts/100 pt-days
~11 alerts per patient-day: the alert-fatigue problem, quantified. Day 30 topic.


In [3]:
imp = json.loads(Path("../models/feature_importance.json").read_text())
top10 = list(imp)[:10]
print("Top-10 by gain: " + top10[0] + ", " + ", ".join(top10[1:5]) + ",")
print("  " + ", ".join(top10[5:]))
print("Read: vitals dominate (Resp/Temp) as physiology suggests; Platelets_miss_24h")
print("in the top 10 vindicates Day 22 (missingness is signal). HospAdmTime + Unit1")
print("ranking high is a shortcut-learning flag: site/admin patterns may proxy for")
print("hospital-specific sepsis rates — validate on the held-out hospital (set B).")

Top-10 by gain: Resp_mean_24h, Temp_mean_6h, Resp_mean_6h, HospAdmTime,
  Unit1, Creatinine_mean_24h, Temp_mean_24h, SBP_mean_6h, HR_mean_6h,
  Platelets_miss_24h
Read: vitals dominate (Resp/Temp) as physiology suggests; Platelets_miss_24h
in the top 10 vindicates Day 22 (missingness is signal). HospAdmTime + Unit1
ranking high is a shortcut-learning flag: site/admin patterns may proxy for
hospital-specific sepsis rates — validate on the held-out hospital (set B).
